# Audited black-hole shadow parameters

This notebook is a readable diagnostic companion to the deterministic table builder.

The canonical inputs are `analysis/CountingBHS_and_Spectra/bhs_source_inputs.csv`, `analysis/CountingBHS_and_Spectra/pesce_theta_g_quantiles.csv`, `analysis/CountingBHS_and_Spectra/bhs_model_sensitivity_checks.csv`, and `analysis/CountingBHS_and_Spectra/bhs_nonadopted_ratio_fits.csv`; `build_bhs_params.py` is the only publication-table calculation path.

## Audited uncertainty treatments

For the six joint fits, transform the paired-chain angular gravitational-radius quantiles directly:

$$\theta_{\rm BHS}=6\sqrt{3}\,\theta_g.$$

For the thirteen fixed-distance fits, write $M=fD$ and cancel the common distance before propagating the remaining fit or model terms:

$$\theta_{\rm BHS}=\frac{6\sqrt{3}\,G(fD)}{Dc^2}=\frac{6\sqrt{3}\,Gf}{c^2}.$$

No production result uses an assumed correlation coefficient of zero or one.

For NGC 4258, the component-tracking/data-selection range is a separate non-probabilistic sensitivity. The builder appends it only after calculating the paired-chain result, 1% non-Keplerian allowance, and adopted quadrature values; it is not combined with or substituted for them.

In [ ]:
from pathlib import Path
import csv
import subprocess
import sys

root = Path.cwd()
if not (root / 'analysis' / 'CountingBHS_and_Spectra').is_dir():
    raise RuntimeError('Run this notebook from the repository root')

builder = root / 'analysis' / 'CountingBHS_and_Spectra' / 'build_bhs_params.py'
output = root / 'analysis' / 'CountingBHS_and_Spectra' / 'BHS_params.csv'
subprocess.run([sys.executable, str(builder)], check=True)

with output.open(newline='') as handle:
    rows = list(csv.DictReader(handle))

print(f'Loaded {len(rows)} audited source rows from {output}')

In [ ]:
from collections import Counter

counts = Counter(row['adopted_uncertainty_model'] for row in rows)
for treatment, count in sorted(counts.items()):
    print(f'{treatment}: {count}')

assert len(rows) == 21
assert not any(row['adopted_uncertainty_model'] in {'correlated', 'uncorrelated'} for row in rows)

In [ ]:
columns = (
    'source_label', 'size_uas', 'size_stat_unc_minus_uas',
    'size_stat_unc_plus_uas', 'size_systematic_unc_minus_uas',
    'size_systematic_unc_plus_uas', 'systematic_status',
    'size_model_check_lower_uas', 'size_model_check_upper_uas',
    'size_model_check_adoption_status',
)
for row in rows:
    print(' | '.join(row.get(column, 'NA') for column in columns))

## Remaining external item

IC 2560 uses Nakai's confirmed 4.106% formal fitting uncertainty and exact shared-distance cancellation. No separately quantified model systematic is available, so the displayed interval remains formal only and the model contribution is unquantified.